# Task 3 – Map Matching

For 1500 Porto taxi trajectories to the road network using leuvenmapmatching.
Output: `matched_1500.csv` (all trips) and `matched_15.csv` (first 15 trips).
Each row = one trip

In [ ]:
# install dependencies if not already installed
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import osmnx
except ImportError:
    install('osmnx')

try:
    import leuvenmapmatching
except ImportError:
    install('leuvenmapmatching')

try:
    import rtree
except ImportError:
    install('rtree')

try:
    import geopandas
except ImportError:
    install('geopandas')

In [2]:
import ast
import json
import os
import warnings

import numpy as np
import pandas as pd
import osmnx as ox
import geopandas as gpd

try:
    from tqdm.notebook import tqdm   
except Exception:
    from tqdm import tqdm           

from leuvenmapmatching.map.inmem import InMemMap
from leuvenmapmatching.matcher.distance import DistanceMatcher

warnings.filterwarnings('ignore')
print('imports ok')

imports ok


In [20]:
#downloading porto graph data

PORTO_BBOX = (-8.73, 41.10, -8.55, 41.19) 
NETWORK_FILE = 'porto_drive.graphml'

if os.path.exists(NETWORK_FILE):
    print('loading saved road network...')
    G = ox.load_graphml(NETWORK_FILE)
else:
    print('downloading Porto road network from OSM...')
    G = ox.graph_from_bbox(
        bbox=PORTO_BBOX,
        network_type='drive',
        simplify=True
    )
    ox.save_graphml(G, NETWORK_FILE)
    print(f'saved to {NETWORK_FILE}')

print(f'nodes: {len(G.nodes):,}  edges: {len(G.edges):,}')

loading saved road network...
nodes: 53,903  edges: 124,814


## Build the InMemMap and Edge ID Table

In [22]:
# get edge GeoDataFrame to work with edge attributes
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)
print('edges_gdf shape:', edges_gdf.shape)
print(edges_gdf.index[:3])  # MultiIndex (u, v, key)

edges_gdf shape: (124814, 16)
MultiIndex([(25503936, 4722746638, 0),
            (25503936,  281726624, 0),
            (25503951, 1243857957, 0)],
           names=['u', 'v', 'key'])


In [23]:
# assign a simple integer road_segment_id to every (u, v, key) edge
edges_gdf = edges_gdf.reset_index()  # u, v, key become columns
edges_gdf['road_segment_id'] = range(len(edges_gdf))

# build lookup: (u, v) -> road_segment_id
# when multiple keys (parallel edges), keep only key=0 (the first / primary edge)
uv_to_id = {}
for _, row in edges_gdf.iterrows():
    uv = (int(row['u']), int(row['v']))
    if uv not in uv_to_id:
        uv_to_id[uv] = int(row['road_segment_id'])

print(f'unique (u,v) edges in lookup: {len(uv_to_id):,}')
edges_gdf[['u', 'v', 'key', 'road_segment_id', 'osmid']].head()

unique (u,v) edges in lookup: 123,520


,u,v,key,road_segment_id,osmid
0,25503936,4722746638,0,0,479127843
1,25503936,281726624,0,1,"[4256507, 1175924414]"
2,25503951,1243857957,0,2,4256513
3,25503951,4507024043,0,3,158906049
4,25504027,1243857999,0,4,"[4256491, 108390772, 479127839]"


In [24]:
# build leuvenmapmatching InMemMap
# nodes: label = osmnx node id, coords = (lat, lon)
print('building InMemMap...')
map_con = InMemMap('porto', use_latlon=True, use_rtree=True, index_edges=True)

for node_id, data in G.nodes(data=True):
    lat = data['y']  # osmnx stores lat as y
    lon = data['x']  # and lon as x
    map_con.add_node(node_id, (lat, lon))

for u, v, _ in G.edges(keys=True):
    map_con.add_edge(u, v)

print('InMemMap built.')
print(f'  nodes: {len(map_con.graph):,}')

building InMemMap...
InMemMap built.
  nodes: 53,903


## Load train-1500 Dataset

In [25]:
CSV_PATH = "train-1500.csv"

df = pd.read_csv(CSV_PATH)
print(df.shape)
print(df.columns.tolist())
df.head(2)

(1500, 9)
['TRIP_ID', 'CALL_TYPE', 'ORIGIN_CALL', 'ORIGIN_STAND', 'TAXI_ID', 'TIMESTAMP', 'DAY_TYPE', 'MISSING_DATA', 'POLYLINE']


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,NaN,NaN,20000589,1372636858,A,False,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303620000596,B,NaN,7.0,20000596,1372637303,A,False,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."


In [26]:
def parse_polyline(s):
    """parse POLYLINE string like '[[lon1,lat1],[lon2,lat2],...]' into list of (lat, lon)."""
    try:
        coords = json.loads(s)
    except (json.JSONDecodeError, TypeError):
        try:
            coords = ast.literal_eval(s)
        except Exception:
            return []
    # Porto dataset is [lon, lat] order
    return [(pt[1], pt[0]) for pt in coords if len(pt) == 2]

# parse all polylines
df['coords'] = df['POLYLINE'].apply(parse_polyline)

# mark trips with fewer than 2 GPS points so the matcher can skip them
valid_mask = df['coords'].apply(len) >= 2
print(f'valid trips: {valid_mask.sum()} / {len(df)}')
df_valid = df[valid_mask].reset_index(drop=True)
df_valid[['TRIP_ID', 'coords']].head(3)

valid trips: 1481 / 1500


,TRIP_ID,coords
0,1372636858620000589,"[(41.141412, -8.618643), (41.141376, -8.618499..."
1,1372637303620000596,"[(41.159826, -8.639847), (41.159871, -8.640351..."
2,1372636951620000320,"[(41.140359, -8.612964), (41.14035, -8.613378)..."


## Map Matching

In [27]:
def match_trip(coords, map_con, uv_to_id,
               max_dist=200, obs_noise=25, dist_noise=25):
    """
    run map matching on one trip.
    returns a list of road_segment_ids (integers), or [] if matching fails.
    """
    if len(coords) < 2:
        return []

    matcher = DistanceMatcher(
        map_con,
        max_dist=max_dist,
        obs_noise=obs_noise,
        dist_noise=dist_noise,
        non_emitting_states=True,
        non_emitting_length_factor=0.75,
        max_lattice_width=5
    )
    try:
        matcher.match(coords, unique=False)
    except Exception:
        return []

    if not matcher.path_pred:
        return []

    # path_pred is a list of (u, v) edge tuples
    edge_ids = []
    seen = set()
    for u, v in matcher.path_pred:
        seg_id = uv_to_id.get((int(u), int(v)))
        # also try reverse direction for undirected matching cases
        if seg_id is None:
            seg_id = uv_to_id.get((int(v), int(u)))
        if seg_id is not None and seg_id not in seen:
            edge_ids.append(seg_id)
            seen.add(seg_id)

    return edge_ids

In [29]:
# keep all 1500 trips in the output, but do not match trips with fewer than 2 points
results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc='map matching'):
    seg_ids = match_trip(row['coords'], map_con, uv_to_id) if len(row['coords']) >= 2 else []
    results.append({
        'trip_index': i,
        'TRIP_ID': row['TRIP_ID'],
        'road_segment_ids': ' '.join(map(str, seg_ids)),
        'num_segments': len(seg_ids)
    })

results_df = pd.DataFrame(results)
print('done.')
print(results_df[['TRIP_ID', 'num_segments']].describe())

map matching:   0%|          | 0/1481 [00:00<?, ?it/s]

done.
            TRIP_ID  num_segments
count  1.481000e+03   1481.000000
mean   1.372661e+18     41.591492
std    1.005289e+13     26.849504
min    1.372637e+18      1.000000
25%    1.372655e+18     24.000000
50%    1.372665e+18     38.000000
75%    1.372669e+18     55.000000
max    1.372675e+18    401.000000


## Export Results

In [30]:
# export all 1500 trips
results_df.to_csv('matched_1500.csv', index=False)
print('saved matched_1500.csv')

# export first 15 trips
matched_15 = results_df.head(15)
matched_15.to_csv('matched_15.csv', index=False)
print('saved matched_15.csv')

print('\nfirst 5 rows of matched_15.csv:')
print(matched_15[['TRIP_ID', 'num_segments', 'road_segment_ids']].to_string())

saved matched_1500.csv
saved matched_15.csv

first 5 rows of matched_15.csv:
                TRIP_ID  num_segments                                                                                                                                                                                                                                                                                                                                                                   road_segment_ids
0   1372636858620000589            23                                                                                                                                                                                                                                                          86441 3142 86442 1036 669 9408 97323 9406 666 9389 24127 4012 5918 3132 1268 1271 86475 57493 57495 95097 90691 1353 1352
1   1372637303620000596            29                                                                    

In [31]:
# save the edge lookup table too so teammates can resolve IDs back to geometry
edges_gdf[['road_segment_id', 'u', 'v', 'key', 'osmid', 'name', 'length', 'geometry']].to_file(
    'porto_edges.gpkg', driver='GPKG'
)
print('saved porto_edges.gpkg (road network with segment IDs)')

saved porto_edges.gpkg (road network with segment IDs)


## Checking how many trips matched

In [32]:
matched = results_df['num_segments'] > 0
print(f'trips with at least 1 segment matched: {matched.sum()} / {len(results_df)}')
print(f'mean segments per matched trip: {results_df.loc[matched, "num_segments"].mean():.1f}')
print(f'trips with no match: {(~matched).sum()}')

trips with at least 1 segment matched: 1481 / 1481
mean segments per matched trip: 41.6
trips with no match: 0


In [33]:
import folium

# pick trip 0's matched segments
trip0_ids = list(map(int, results_df.iloc[0]['road_segment_ids'].split()))

# get the edges for those segments
trip0_edges = edges_gdf[edges_gdf['road_segment_id'].isin(trip0_ids)]

# plot on a map
m = folium.Map(location=[41.15, -8.62], zoom_start=14)
for _, edge in trip0_edges.iterrows():
    coords = [(lat, lon) for lon, lat in edge['geometry'].coords]
    folium.PolyLine(coords, color='blue', weight=3).add_to(m)

# also plot the raw GPS points
raw_coords = df_valid.iloc[0]['coords']  # list of (lat, lon)
for lat, lon in raw_coords:
    folium.CircleMarker([lat, lon], radius=3, color='red').add_to(m)

m.save('trip0_check.html')
print('saved trip0_check.html')

saved trip0_check.html


In [35]:
# 
seg_map = edges_gdf[['road_segment_id', 'u', 'v']].copy()
seg_map.columns = ['id', 'u', 'v']
seg_map.to_csv('segment_id_map_p2.csv', index=False)
print(f"exported {len(seg_map)} segment ID mappings")

exported 124814 segment ID mappings


In [37]:
import pandas as pd

robust = pd.read_csv('leuven_1500_robust.csv')
seg_map = pd.read_csv('segment_id_map_p2.csv')  

uv_to_seg = {(int(r['u']), int(r['v'])): int(r['id']) for _, r in seg_map.iterrows()}

# convert one trip and count unmapped edges
trip0_nodes = [int(x) for x in robust.iloc[0]['nodes'].split(',')]
unmapped = 0
for u, v in zip(trip0_nodes, trip0_nodes[1:]):
    if (u, v) not in uv_to_seg and (v, u) not in uv_to_seg:
        unmapped += 1

total_edges = len(trip0_nodes) - 1
print(f"Trip 0: {total_edges} edges, {unmapped} unmapped ({unmapped/total_edges*100:.1f}%)")

Trip 0: 23 edges, 0 unmapped (0.0%)


In [40]:
#unmapped edges
total_unmapped = 0
total_edges = 0
for _, r in robust.iterrows():
    if pd.isna(r['nodes']) or str(r['nodes']).strip() == '':
        continue
    nodes = [int(x) for x in str(r['nodes']).split(',') if x.strip()]
    for u, v in zip(nodes, nodes[1:]):
        total_edges += 1
        if (u, v) not in uv_to_seg and (v, u) not in uv_to_seg:
            total_unmapped += 1

print(f"Overall: {total_unmapped} / {total_edges} edges unmapped ({total_unmapped/total_edges*100:.1f}%)")

Overall: 424 / 75292 edges unmapped (0.6%)


In [41]:
# load final matched_1500.csv
final = pd.read_csv('matched_1500.csv')
seg_map = pd.read_csv('segment_id_map_p2.csv')

matched = final['num_segments'] > 0
print(f"1. Trips matched: {matched.sum()} / {len(final)}")
print(f"2. Mean segments: {final[matched].num_segments.mean():.1f}")
print(f"3. Max segment ID: {max(int(x) for ids in final.road_segment_ids.dropna() for x in ids.split())}")
print(f"4. seg_map covers all IDs: {max(int(x) for ids in final.road_segment_ids.dropna() for x in ids.split()) < len(seg_map)}")
print(f"5. Trip 0 IDs: {final.iloc[0]['road_segment_ids'][:50]}...")

1. Trips matched: 1481 / 1481
2. Mean segments: 41.6
3. Max segment ID: 124748
4. seg_map covers all IDs: True
5. Trip 0 IDs: 86441 3142 86442 1036 669 9408 97323 9406 666 9389...
